# 03 — The Causal-Probe Gap

The headline experiment. For each residual site $k$ we measure:

$$D_{\text{subj}}(k) = \text{probe macro-F1 at the last subject token}$$
$$D_{\text{read}}(k) = \text{probe macro-F1 at the final prompt token}$$
$$C(k) = \text{mean subject-swap patch restoration}$$

and derive:

- **CPG$_{\text{subj}}(k) = D_{\text{subj}}(k) - C(k)$** — *dormant knowledge*: the fact is
  linearly decodable at the subject token but no longer causally used there.
- **Readout onset** — first site where $D_{\text{read}}$ clears chance by a margin:
  the answer has become linearly accessible at the position the model reads out from.
- **Routing offset** — first site where $C$ falls under the patch margin: the subject
  token has stopped being causally necessary (attention has already moved the fact).
- **Handoff window** = routing offset − readout onset: how many sites the network
  spends "in transit".

Empirically on GPT-2: $D_{\text{subj}}$ sits near ceiling from the embedding site
(identity is linearly present early); $D_{\text{read}}$ rises late; $C$ decays at
roughly the same late sites — the handoff — after which CPG$_{\text{subj}}$ stays
high: dormant knowledge in the tail of the network.


In [ ]:
from factlens.config import DotDict
from factlens.pipelines.context import make_context
from factlens.pipelines.cpg_experiment import run_cpg

cfg = DotDict({
    'seed': 13, 'device': 'cpu', 'output_root': 'results',
    'model': {'name_or_path': 'gpt2', 'display_name': 'GPT-2 (124M)',
              'slug': 'gpt2', 'dtype': 'float32', 'prepend_bos': False},
    'data': {'relations': None, 'limit_per_relation': None,
             'known_only': True, 'known_rank_threshold': 5},
    'lens': {'align': 'center', 'topk': [5, 10, 100]},
    'probe': {'positions': ['subject_end', 'prompt_end'], 'test_template_idx': -1,
              'val_fraction': 0.25, 'epochs': 300, 'lr': 0.001,
              'weight_decay': 0.0001, 'batch_size': 64, 'patience': 30,
              'bias': True},
    'patch': {'positions': 'subject_end', 'sites': 'residual', 'max_facts': None},
    'cpg': {'probe_margin': 0.15, 'patch_margin': 0.15, 'run_length': 2},
    'tracking': {'tensorboard': False},
    'publish': {'figures': False},
})
# Full bank: ~5-8 min on CPU, ~1-2 min on GPU. Use limit_per_relation=6
# for a quicker first pass (onsets become noisier).
ctx = make_context(cfg, run_name='nb03_cpg')
print(ctx.dataset.describe())


In [ ]:
out = run_cpg(ctx, progress=True)


In [ ]:
from factlens.analysis.cpg import cpg_to_markdown
from IPython.display import Markdown

Markdown(cpg_to_markdown(out['cpg'], model_name='GPT-2 (124M)'))

In [ ]:
from factlens.analysis.figures import plot_cpg

fig = plot_cpg(out['cpg'], model_name='GPT-2 (124M)')
fig

## Reading the three panels

**Panel 1 — Decodability.** $D_{\text{subj}}$ (blue) sits near ceiling from the
embedding site: subject identity — and hence its associated object — is trivially
linearly present at the subject token. The dashed controls show how much of that
is surface cue rather than learned association (the Hewitt-Liang control at the
subject position is *also* largely solvable — an honest caveat). $D_{\text{read}}$
(green) is the informative curve: near chance until the last third of the stack,
then rising — the fact becomes linearly accessible at the readout position only
after attention routes it there.

**Panel 2 — Causality.** $C(k)$ is ~1 from the embedding site (swapping the
subject token's embedding *is* swapping the fact) and decays over the last
blocks: once routing completes, the subject token's late-layer representation no
longer matters. The red line marks the routing offset.

**Panel 3 — Dormant knowledge.** CPG$_{\text{subj}} = D_{\text{subj}} - C$ grows
through the handoff window (shaded) and stays high afterwards: the fact remains
decodable at the subject token long after the model stopped using it.


In [ ]:
print('Per-relation handoff statistics (where defined):')
print(f"{'relation':<18} {'D_read-onset':>12} {'C-offset':>9} {'handoff':>8}")
print('-' * 50)
for rel, stats in out['cpg_per_relation'].items():
    print(f"{rel:<18} {str(stats['readout_onset']):>12} "
          f"{str(stats['routing_offset']):>9} "
          f"{str(stats['handoff_window_sites']):>8}")


## A positional view: causal tracing on one fact

Subject-swap patching fixes the position (last subject token) and sweeps
sites. Causal tracing does the opposite sweep — every position, at every
site — under ROME-style embedding noise, producing the classic indirect-effect
heatmap:

In [ ]:

def dataset_example(ctx, relation, subject, template_idx=0):
    return ctx.dataset.example(relation, subject, template_idx)
from factlens.causality.tracing import CausalTracer
from factlens.analysis.figures import plot_trace_heatmap

tracer = CausalTracer(ctx.model, ctx.module_map, noise_std_mult=3.0)
tracer.attach_tokenizer(ctx.tokenizer)

ex = dataset_example(ctx, 'capital-country', 'France')
trace = tracer.trace_example(ex, ctx.dataset.counterfactual(ex), seed=13)
if trace is not None:
    fig = plot_trace_heatmap(trace, model_name='GPT-2 (124M)')
else:
    print('model failed the direction check for this fact; pick another')

def dataset_example(ctx, relation, subject, template_idx=0):
    return ctx.dataset.example(relation, subject, template_idx)

## Interpretation checklist

1. **Identity is cheap; association is the story.** $D_{\text{subj}} \approx 1$
   everywhere mostly reflects subject identity — that is why the control curve
   matters, and why $D_{\text{read}}$ (where identity is *not* directly available)
   is the decodability curve that tracks routing.
2. **C decays late.** The routing window (where patching stops restoring)
   localizes the attention move from subject to readout position.
3. **The handoff window is where D_read rises while C decays.** The two events
   are close in GPT-2 — once the fact is routed, the subject token is released
   quickly.
4. **The dormant tail is large.** After the routing offset, CPG$_{\text{subj}}$
   stays high for the rest of the stack — consistent with FFN layers holding
   key-value associations that the readout no longer consults at the subject
   position (Geva et al. 2021/2022).

## Extensions

- `--model pythia-410m` / `--model qwen2.5-0.5b`: same curves, RMSNorm +
  different depth — does the dormant tail scale with depth?
- `patch.sites: residual+sublayers`: separate attention from MLP
  contributions to C(k).
- Replace the mean-centered lens with a *learned* tuned lens (Belrose et al.
  2023) and check whether the lens trajectory onset moves earlier.
- Probe with `bias=False` direction-only probes and compare geometry.
